# OmniVoice Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k2-fsa/OmniVoice/blob/master/docs/OmniVoice.ipynb)

This notebook demonstrates the basic usage of [OmniVoice](https://github.com/k2-fsa/OmniVoice), a massively multilingual zero-shot TTS model supporting 600+ languages.

**Contents:**
1. Installation
2. Option A — Gradio Demo (interactive web UI, no code needed)
3. Option B — Python API
   - 3.1 Load Model
   - 3.2 Voice Cloning
   - 3.3 Voice Design
   - 3.4 Auto Voice

In [ ]:
# 1. Install dependencies
!pip install omnivoice omnivoice-server pyngrok

# 2. Setup Ngrok (Ganti 'YOUR_TOKEN' dengan Authtoken dari dashboard ngrok.com)
from pyngrok import ngrok
NGROK_AUTH_TOKEN = "30RF7rmQ09YRusehjvmnu4tGXZW_qy5HBxPZEryHDDZ4D54x"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# 3. Jalankan OmniVoice Server dan tampilkan log
import subprocess
import sys
import threading

# Port default omnivoice-server adalah 8000
# Paksa pakai CUDA agar kencang (GPU) dan aktifkan streaming
cmd = ["omnivoice-server", "--port", "8000", "--num-step", "12", "--stream", "--device", "cuda"]
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

def log_reader(proc):
    for line in iter(proc.stdout.readline, ""):
        print(f"[SERVER LOG] {line.strip()}")

# Jalankan log reader di thread terpisah agar tidak memblokir cell
thread = threading.Thread(target=log_reader, args=(process,), daemon=True)
thread.start()

print("Sedang menyalakan server, tunggu sebentar...")
time.sleep(15)

# 4. Buka Tunnel Ngrok
public_url = ngrok.connect(8000).public_url
print(f"\n[SUKSES] Copy URL ini ke config.py Anda:\n{public_url}")

# Menjaga notebook tetap hidup
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    process.terminate()
    print("Server dimatikan.")


Sedang menyalakan server, tunggu sebentar...
[SERVER LOG] INFO:     Started server process [13535]
[SERVER LOG] INFO:     Waiting for application startup.
[SERVER LOG] 2026-05-11T20:53:13Z [INFO ] [omnivoice_server.app] omnivoice-server starting up...
[SERVER LOG] 2026-05-11T20:53:13Z [INFO ] [omnivoice_server.app]   device=cuda  num_step=12  max_concurrent=2
[SERVER LOG] 2026-05-11T20:53:19Z [INFO ] [numexpr.utils] NumExpr defaulting to 2 threads.
[SERVER LOG] 2026-05-11T20:53:20Z [INFO ] [omnivoice_server.services.model] Loading model 'k2-fsa/OmniVoice' on cuda...
[SERVER LOG] 
[SERVER LOG] Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 17012.78it/s]
[SERVER LOG] 
[SERVER LOG] Loading weights: 100%|██████████| 313/313 [00:01<00:00, 195.43it/s]
[SERVER LOG] 
[SERVER LOG] Loading weights: 100%|██████████| 527/527 [00:00<00:00, 1422.54it/s]
[SERVER LOG] 2026-05-11T20:53:25Z [INFO ] [omnivoice_server.services.model] Model loaded in 5.2s. RAM: 839MB -> 2358MB (+1520MB)
[SERVER LO

## 1. Installation

Colab already provides a compatible PyTorch + CUDA environment, so we only need to install OmniVoice.

In [ ]:
!pip install omnivoice

## 2. Option A — Gradio Demo

Launch an interactive web UI with a public Gradio link. The `--share` flag creates a temporary public URL so you can access the demo from any browser.

> **If you prefer to use the Python API directly, skip to Option B below.**

In [ ]:
!omnivoice-demo --share

## 3. Option B — Python API

### 3.1 Load Model

In [6]:
from omnivoice import OmniVoice
import soundfile as sf
import torch
from IPython.display import Audio, display

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=True,
)

KeyboardInterrupt: 

### 3.2 Voice Cloning

Clone a voice from a short (3-10s) reference audio clip. Upload your own `ref.wav` or use any audio file.

`ref_text` is optional — if omitted, the model uses Whisper ASR to auto-transcribe it.

In [7]:
from google.colab import files

print("Upload a reference audio file (wav/mp3/flac):")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {ref_audio_path}")

Upload a reference audio file (wav/mp3/flac):


Saving loli.mp3 to loli.mp3
Uploaded: loli.mp3


In [8]:
# Jalankan ini di Google Colab
import os

# Kita namakan profilnya "hina" saja ya biar cocok sama nama asistennya
os.makedirs('/root/.local/share/omnivoice/profiles/hina', exist_ok=True)

# Pindahkan loli.mp3 ke dalam folder profil hina
!cp /content/loli.mp3 /root/.local/share/omnivoice/profiles/hina/ref_audio.wav

print("Selesai! Hina sekarang sudah punya suara baru.")


Selesai! Hina sekarang sudah punya suara baru.


In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice cloning.",
    ref_audio=ref_audio_path,
    # ref_text="Transcription of the reference audio.",  # optional
)

sf.write("clone_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.3 Voice Design

Describe the desired voice with speaker attributes — no reference audio needed.

Supported attributes: gender, age, pitch, style (whisper), English accent, Chinese dialect. See [docs/voice-design.md](https://github.com/k2-fsa/OmniVoice/blob/master/docs/voice-design.md) for the full list.

In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice design.",
    instruct="female, low pitch, british accent",
)

sf.write("design_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.4 Auto Voice

Let the model choose a voice automatically — no reference audio or instruct needed.

In [ ]:
audio = model.generate(
    text="This is a sentence generated with automatic voice selection.",
)

sf.write("auto_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))